# Ecosystem Runtime + Eval Journey

Journey Name: Ecosystem Runtime + Eval
Journey Purpose: Show the end-to-end `llm_client + prompt_eval` substrate journey, prove the shared observability loop, and keep the next phases explicit through live or provisional artifacts.
Notebook Mode: mixed
Related Docs:
- `llm_client/docs/ECOSYSTEM_TOP_DOWN_ARCHITECTURE.md`
- `llm_client/docs/adr/0010-cross-project-runtime-substrate.md`
- `llm_client/docs/adr/0011-prompt-assets-explicit-identity.md`
- `prompt_eval/docs/adr/0002-prompt-eval-run-family-mapping.md`
- `project-meta/docs/ops/JUPYTER_NOTEBOOK_SHORT_TERM_RULES.md`
Related Code:
- `llm_client/llm_client/observability/experiments.py`
- `llm_client/llm_client/prompts.py`
- `llm_client/llm_client/prompt_assets.py`
- `prompt_eval/prompt_eval/runner.py`
- `prompt_eval/prompt_eval/query.py`
- `prompt_eval/prompt_eval/prompt_assets.py`
Related Tests:
- `llm_client/tests/test_experiment_log.py`
- `llm_client/tests/test_prompts.py`
- `prompt_eval/tests/test_runner.py`
- `prompt_eval/tests/test_query.py`
- `prompt_eval/tests/test_prompt_assets.py`
Related Evidence:
- `llm_client/docs/adr/0010-cross-project-runtime-substrate.md`
- `llm_client/docs/adr/0011-prompt-assets-explicit-identity.md`
- `prompt_eval/docs/adr/0002-prompt-eval-run-family-mapping.md`


## How To Use This Notebook

- Run top to bottom inside a Python environment that can import both `llm_client` and `prompt_eval`.
- The phase contracts are canonical in `project-meta/notebooks/notebook_registry.yaml`; this notebook renders them in runnable form.
- Phase 0 and Phase 1 run real code.
- Later phases are still provisional, but they emit explicit stub artifacts so the journey stays continuous without hidden state.
- This notebook is mixed use: it contains real proof for proven slices and planning-mode stubs for the rest.

Notebook path assumptions:
- `~/projects/llm_client`
- `~/projects/prompt_eval`
- `~/projects/project-meta`


In [ ]:
import asyncio
import logging
import sys
import tempfile
from pathlib import Path
from pprint import pprint
from unittest.mock import AsyncMock, patch

PROJECTS = Path.home() / "projects"
for repo in (PROJECTS / "llm_client", PROJECTS / "prompt_eval"):
    repo_str = str(repo)
    if repo_str not in sys.path:
        sys.path.insert(0, repo_str)

logging.getLogger("llm_client.io_log").setLevel(logging.ERROR)
logging.getLogger("prompt_eval.runner").setLevel(logging.ERROR)


In [1]:
current_state = {
    "completed_slices": [
        "top-down ecosystem architecture and ADR set",
        "prompt_eval -> llm_client dual-write for runs and items",
        "read-side reload from shared observability by execution_id",
        "shared family-level aggregates for corpus evaluator outputs",
        "minimal explicit prompt asset layer with prompt_ref resolution",
    ],
    "shared_backend_now_supports": [
        "llm/embedding call logging",
        "experiment runs and items",
        "family-level experiment aggregates",
        "cross-project query surfaces",
        "prompt asset provenance in observability",
    ],
    "still_intentionally_open": [
        "real cross-project prompt family migration",
        "shared data plane for datasets and artifacts",
        "higher-level cohort analytics UX",
        "workflow/runtime layer boundary",
        "repo/package topology end state",
    ],
}
pprint(current_state)


{'completed_slices': ['top-down ecosystem architecture and ADR set',
                      'prompt_eval -> llm_client dual-write for runs and items',
                      'read-side reload from shared observability by execution_id',
                      'shared family-level aggregates for corpus evaluator outputs'],
 'shared_backend_now_supports': ['llm/embedding call logging',
                                 'experiment runs and items',
                                 'family-level experiment aggregates',
                                 'cross-project query surfaces'],
 'still_intentionally_open': ['shared prompt asset library',
                              'shared data plane for datasets and artifacts',
                              'higher-level cohort analytics UX',
                              'workflow/runtime layer boundary',
                              'repo/package topology end state']}


## Phase Map

These phases are intentionally acceptance-oriented. The point is not to build a huge system all at once; it is to prove each layer on the smallest real slice before moving on.


In [2]:
roadmap = [
    {
        "phase": 0,
        "name": "Runtime / Eval Substrate",
        "status": "proven",
        "execution_mode": "live",
        "objective": "Make llm_client the shared execution + observability substrate for prompt_eval.",
        "smallest_slice": "dual-write prompt_eval runs/items, reload by execution_id, persist corpus aggregates.",
    },
    {
        "phase": 1,
        "name": "Prompt Asset Layer",
        "status": "partial",
        "execution_mode": "live",
        "objective": "Replace prompt drift with explicit prompt asset identity and deterministic resolution.",
        "smallest_slice": "shared prompt refs, manifest validation, and PromptVariant materialization from prompt_ref.",
    },
    {
        "phase": 2,
        "name": "Shared Data Plane",
        "status": "planned",
        "execution_mode": "stub",
        "objective": "Standardize dataset/artifact/provenance contracts across projects.",
        "smallest_slice": "one registry shape for dataset IDs and artifact lineage.",
    },
    {
        "phase": 3,
        "name": "Analytics And Query UX",
        "status": "planned",
        "execution_mode": "stub",
        "objective": "Turn shared runs and aggregates into useful analyst-facing reports.",
        "smallest_slice": "one family report shape driven by execution_id plus provenance.",
    },
    {
        "phase": 4,
        "name": "Evaluation Expansion Beyond Prompt Eval",
        "status": "planned",
        "execution_mode": "stub",
        "objective": "Use the same backend for non-prompt experiments without collapsing boundaries.",
        "smallest_slice": "one non-prompt family contract using the same run/item/aggregate rails.",
    },
    {
        "phase": 5,
        "name": "Workflow / Runtime Layer",
        "status": "planned",
        "execution_mode": "stub",
        "objective": "Keep orchestration above llm_client and prompt_eval.",
        "smallest_slice": "one workflow contract that reuses llm_client observability instead of forking it.",
    },
]
pprint(roadmap)


Phase 0: Runtime/eval substrate proven [done]
  Objective: Make llm_client the shared execution + observability substrate for prompt_eval.
  Smallest slice: dual-write prompt_eval runs/items, reload by execution_id, persist corpus aggregates.
  Exit criteria:
    - shared runs are authoritative for prompt_eval
    - read path reconstructs EvalResult
    - corpus metrics round-trip through the shared backend

Phase 1: Prompt asset layer [next]
  Objective: Replace inline-message drift with explicit prompt asset identity and lineage.
  Smallest slice: shared prompt refs plus one asset resolver for canonical prompts.
  Exit criteria:
    - every promoted prompt has a stable asset id
    - derivation is explicit metadata, not override magic
    - observability records prompt asset refs directly

Phase 2: Shared data plane [next]
  Objective: Standardize datasets, artifacts, and lineage across projects.
  Smallest slice: dataset registry + artifact registry + run/artifact links.
  Exit crit

## Phase 0: Runtime / Eval Substrate

Purpose: Prove that `prompt_eval` writes authoritative runs, items, and aggregates into `llm_client`, and that the shared backend can reconstruct an experiment family.
Input -> Output: `prompt_eval_experiment_invocation -> shared_observability_family`
Acceptance Criteria:
- Shared runs are authoritative for `prompt_eval`.
- Read-side reconstruction returns the expected `EvalResult` family.
- Corpus-level aggregate metrics round-trip through the shared backend.
Status: proven
Execution Mode: live

Promotion Path:
- Keep this phase live and stable while later phases advance behind explicit contracts.


## Executable Demo: Current Shared Observability Loop

This cell proves the current state with no real model calls:
- `prompt_eval.run_experiment()` emits shared runs and items into `llm_client`,
- corpus evaluator results emit shared family-level aggregates,
- `prompt_eval.load_result_from_observability()` reconstructs the family from shared storage.


In [3]:
from llm_client import LLMCallResult
from llm_client.observability import (
    configure_logging,
    get_experiment_aggregates,
    get_runs,
)
from prompt_eval import (
    Experiment,
    ExperimentInput,
    PromptEvalObservabilityConfig,
    PromptVariant,
    load_result_from_observability,
    run_experiment,
)
from prompt_eval.evaluators import EvalScore

demo_root = Path(tempfile.mkdtemp(prefix="ecosystem_notebook_"))
demo_project = "ecosystem_notebook_demo"
demo_dataset = "ecosystem_demo"
demo_execution_id = "demo_exec_001"

configure_logging(
    enabled=True,
    data_root=demo_root,
    project=demo_project,
    db_path=demo_root / f"{demo_project}.db",
)

demo_experiment = Experiment(
    name=demo_dataset,
    variants=[
        PromptVariant(
            name="baseline",
            messages=[{"role": "user", "content": "Summarize: {input}"}],
        ),
        PromptVariant(
            name="candidate",
            prompt_ref="shared.summarize.concise@1",
            messages=[{"role": "user", "content": "Summarize crisply: {input}"}],
        ),
    ],
    inputs=[
        ExperimentInput(id="doc1", content="The quick brown fox."),
        ExperimentInput(id="doc2", content="Jumps over the lazy dog."),
    ],
    n_runs=1,
)

demo_observability = PromptEvalObservabilityConfig(
    project=demo_project,
    dataset=demo_dataset,
    experiment_execution_id=demo_execution_id,
)

def demo_corpus_eval(outputs):
    return EvalScore(
        score=0.6,
        dimension_scores={"coverage": 0.8, "redundancy": 0.4},
        reasoning="Broad enough but somewhat repetitive.",
    )

async def run_demo():
    with patch("prompt_eval.runner.acall_llm", new_callable=AsyncMock) as mock_llm:
        mock_llm.return_value = LLMCallResult(
            content="summary text",
            usage={"total_tokens": 50},
            cost=0.001,
            model="test-model",
        )
        result = await run_experiment(
            demo_experiment,
            evaluator=lambda output, expected: 0.85,
            corpus_evaluator=demo_corpus_eval,
            observability=demo_observability,
        )

    reloaded = load_result_from_observability(
        demo_execution_id,
        project=demo_project,
        dataset=demo_dataset,
    )
    runs = get_runs(project=demo_project, dataset=demo_dataset, limit=10)
    aggregates = get_experiment_aggregates(
        project=demo_project,
        dataset=demo_dataset,
        family_id=demo_execution_id,
        limit=10,
    )
    return {
        "execution_id": result.execution_id,
        "variants": result.variants,
        "trial_count": len(result.trials),
        "shared_run_count": len(runs),
        "aggregate_count": len(aggregates),
        "baseline_mean_score": reloaded.summary["baseline"].mean_score,
        "candidate_mean_score": reloaded.summary["candidate"].mean_score,
        "baseline_corpus_score": reloaded.summary["baseline"].corpus_score,
        "candidate_corpus_dimensions": reloaded.summary["candidate"].corpus_dimension_scores,
        "aggregate_types": sorted({row["aggregate_type"] for row in aggregates}),
    }

demo_summary = asyncio.run(run_demo())
pprint(demo_summary)


{'aggregate_count': 2,
 'aggregate_types': ['prompt_eval.corpus_evaluator'],
 'baseline_corpus_score': 0.6,
 'baseline_mean_score': 0.85,
 'candidate_corpus_dimensions': {'coverage': 0.8, 'redundancy': 0.4},
 'candidate_mean_score': 0.85,
 'execution_id': 'demo_exec_001',
 'shared_run_count': 2,
 'trial_count': 4,
 'variants': ['baseline', 'candidate']}


## Executable Demo: Inspect The Raw Shared Records

This view shows the actual shape of what the shared backend now contains for the demo family.


In [4]:
raw_backend_view = {
    "runs": [
        {
            "condition_id": row["condition_id"],
            "replicate": row["replicate"],
            "n_items": row["n_items"],
            "avg_score": row["summary_metrics"]["avg_score"],
            "total_tokens": row["summary_metrics"]["total_tokens"],
        }
        for row in sorted(
            get_runs(project=demo_project, dataset=demo_dataset, limit=10),
            key=lambda row: (row["condition_id"], row["replicate"]),
        )
    ],
    "aggregates": [
        {
            "aggregate_type": row["aggregate_type"],
            "condition_id": row["condition_id"],
            "metrics": row["metrics"],
        }
        for row in sorted(
            get_experiment_aggregates(
                project=demo_project,
                dataset=demo_dataset,
                family_id=demo_execution_id,
                limit=10,
            ),
            key=lambda row: row["condition_id"],
        )
    ],
}
pprint(raw_backend_view)


{'aggregates': [{'aggregate_type': 'prompt_eval.corpus_evaluator',
                 'condition_id': 'baseline',
                 'metrics': {'coverage': 0.8,
                             'redundancy': 0.4,
                             'score': 0.6}},
                {'aggregate_type': 'prompt_eval.corpus_evaluator',
                 'condition_id': 'candidate',
                 'metrics': {'coverage': 0.8,
                             'redundancy': 0.4,
                             'score': 0.6}}],
 'runs': [{'avg_score': 85.0,
           'condition_id': 'baseline',
           'n_items': 2,
           'replicate': 0,
           'total_tokens': 100},
          {'avg_score': 85.0,
           'condition_id': 'candidate',
           'n_items': 2,
           'replicate': 0,
           'total_tokens': 100}]}


## Phase 1: Prompt Asset Layer

Purpose: Introduce explicit prompt asset identity and deterministic resolution so reusable prompts can be referenced across projects without hidden overrides.
Input -> Output: `prompt_asset_reference -> rendered_prompt_variant`
Acceptance Criteria:
- Each promoted prompt has one explicit `prompt_ref`.
- Prompt manifests resolve deterministically and fail loudly on drift.
- `prompt_eval` can materialize a `PromptVariant` from `prompt_ref`.
Status: partial
Execution Mode: live

Promotion Path:
- The next proof step is migrating one real cross-project prompt family to shared `prompt_ref`s.


In [ ]:
from llm_client import load_prompt_asset, render_prompt
from prompt_eval import build_prompt_variant_from_ref

phase_1_manifest = load_prompt_asset("shared.summarize.concise@1")
phase_1_messages = render_prompt(prompt_ref="shared.summarize.concise@1", style="bullet")
phase_1_variant = build_prompt_variant_from_ref(
    name="shared_bullet_variant",
    prompt_ref="shared.summarize.bullet@1",
    render_context={"bullet_count": 3},
)
phase_1_artifact = {
    "manifest": phase_1_manifest.model_dump(),
    "rendered_messages": phase_1_messages,
    "variant": phase_1_variant.model_dump(),
}
pprint(phase_1_artifact)


## Phase 2: Shared Data Plane

Purpose: Standardize datasets, artifacts, and provenance across projects without forcing every domain payload into the observability database.
Input -> Output: `canonical_dataset_reference -> dataset_and_artifact_registry_links`
Acceptance Criteria:
- Runs reference canonical dataset identifiers.
- Large payloads live as artifacts, not observability row blobs.
- Cross-project lineage is queryable through stable IDs.
Status: planned
Execution Mode: stub

Promotion Path:
- Replace the stub artifact below with a real dataset/artifact registry slice.


In [ ]:
phase_2_artifact = {
    "mode": "stub",
    "dataset": {
        "dataset_id": "news_entities@3",
        "schema": "text_dataset_v1",
    },
    "artifact": {
        "artifact_id": "artifact://research_v3/prompt_eval_result/demo",
        "kind": "prompt_eval_result",
    },
    "lineage": {
        "prompt_ref": phase_1_artifact["variant"]["prompt_ref"],
        "source_execution_id": demo_execution_id,
    },
}
pprint(phase_2_artifact)


## Phase 3: Analytics And Query UX

Purpose: Turn shared runs, items, and aggregates into analyst-facing reports instead of leaving the backend as a raw storage layer.
Input -> Output: `shared_observability_family -> analyst_facing_family_and_cohort_report`
Acceptance Criteria:
- One `execution_id` is enough to reconstruct a useful family report.
- Cross-project comparisons do not require ad hoc SQL.
- Provenance links are visible in the analysis surface.
Status: planned
Execution Mode: stub

Promotion Path:
- Replace the stub report with one real family/cohort report helper.


In [ ]:
phase_3_artifact = {
    "mode": "stub",
    "family_report": {
        "execution_id": demo_execution_id,
        "dataset_id": phase_2_artifact["dataset"]["dataset_id"],
        "prompt_ref": phase_2_artifact["lineage"]["prompt_ref"],
        "sections": ["summary", "variant_comparison", "provenance"],
    },
}
pprint(phase_3_artifact)


## Phase 4: Evaluation Expansion Beyond Prompt Eval

Purpose: Reuse the shared observability contract for non-prompt experiments without turning `prompt_eval` into a generic catch-all experimentation package.
Input -> Output: `non_prompt_workload_definition -> shared_non_prompt_experiment_family`
Acceptance Criteria:
- Non-prompt experiments can write runs, items, and aggregates to the shared backend.
- `prompt_eval` remains prompt-specific.
- Shared reporting helpers work across experiment types.
Status: planned
Execution Mode: stub

Promotion Path:
- Replace the stub family below with one real non-prompt workload using the same backend contract.


In [ ]:
phase_4_artifact = {
    "mode": "stub",
    "workload": "retrieval_ablation",
    "family": {
        "execution_id": "retrieval_ablation_demo_001",
        "report_dependency": phase_3_artifact["family_report"]["sections"],
        "backend_contract": ["runs", "items", "aggregates"],
    },
}
pprint(phase_4_artifact)


## Phase 5: Workflow / Runtime Layer

Purpose: Keep orchestration above `llm_client` and `prompt_eval` so durable workflows, resumes, and checkpoints do not collapse the substrate boundary.
Input -> Output: `workflow_definition -> orchestrated_run_family`
Acceptance Criteria:
- Workflow state and resume are proven on a real task.
- `llm_client` still owns calls and observability.
- The workflow layer does not fork its own logging stack.
Status: planned
Execution Mode: stub

Promotion Path:
- Replace the stub workflow contract below with one real orchestrated workload that reuses the shared rails.


In [ ]:
phase_5_artifact = {
    "mode": "stub",
    "workflow": {
        "name": "document_review",
        "runtime": "langgraph",
        "evaluation_dependency": phase_4_artifact["family"]["backend_contract"],
        "observability_runtime": "llm_client",
    },
}
pprint(phase_5_artifact)


## Repo / Package Topology

This should be a late decision, not an early aesthetic one.


In [ ]:
topology_options = [
    {
        "option": "separate repos, shared packages",
        "recommended_now": True,
        "why": "keeps boundaries clear while letting llm_client remain the substrate",
        "risk": "cross-repo drift if release discipline is weak",
    },
    {
        "option": "monorepo with multiple packages",
        "recommended_now": False,
        "why": "reduces drift and makes cross-package changes easier",
        "risk": "can blur boundaries if package ownership is not explicit",
    },
    {
        "option": "merge prompt_eval into llm_client package",
        "recommended_now": False,
        "why": "fewest moving parts on paper",
        "risk": "collapses the boundary between runtime substrate and prompt-eval product logic",
    },
]
pprint(topology_options)


## Decision Gates

These are the conditions under which we should build, defer, or stop.


In [ ]:
decision_gates = {
    "build_now": [
        "prompt asset identity and discovery",
        "shared dataset/artifact registry",
        "query/report ergonomics around execution_id and cohorts",
    ],
    "defer_until_needed": [
        "full workflow engine rewrite",
        "repo merger",
        "global prompt override mechanism",
    ],
    "stop_if": [
        "shared abstractions do not reduce per-project plumbing",
        "new layers duplicate existing library functionality without adding leverage",
        "agents cannot tell which prompt, dataset, or artifact was actually used",
    ],
}

next_slices = [
    "Phase 1a: define prompt asset manifest schema and one shared prompt library slice",
    "Phase 1b: let prompt_eval variants resolve from prompt_ref without inline messages",
    "Phase 2a: define dataset + artifact registry schema",
    "Phase 2b: link prompt_eval JSON artifacts into the shared registry",
    "Phase 3a: add execution_id cohort report helpers",
]

pprint({"decision_gates": decision_gates, "next_slices": next_slices})
